<a href="https://colab.research.google.com/github/angioitoan2409/flood_forecasting/blob/main/radolan_events.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
OUT_DIR = "/content/drive/MyDrive/flood_forecasting/model_inputs"

In [10]:
import urllib.request
import gzip
import tarfile
import io
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin, rowcol
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS
from datetime import datetime

# ---- Config (uses your existing OUT_DIR from the cell above) ----
CACHE_DIR = "/content/radolan_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

RADOLAN_CRS = CRS.from_proj4(
    "+proj=stere +lat_0=90 +lat_ts=60 +lon_0=10 "
    "+a=6370040 +b=6370040 +x_0=0 +y_0=0 +units=m +no_defs"
)

# ---- Download + extract the one test timestep (2021-07-17 08:05) ----
year, month, day = 2021, 7, 17
monthly_url = f"https://opendata.dwd.de/climate_environment/CDC/grids_germany/5_minutes/radolan/reproc/2017_002/asc/{year}/YW2017.002_{year}{month:02d}_asc.tar"
monthly_path = f"{CACHE_DIR}/YW2017.002_{year}{month:02d}_asc.tar"

if not os.path.exists(monthly_path):
    print("Downloading monthly archive...")
    urllib.request.urlretrieve(monthly_url, monthly_path)

daily_name = f"YW2017.002_{year}{month:02d}{day:02d}_asc.tar.gz"
with tarfile.open(monthly_path) as tar:
    daily_bytes = tar.extractfile(daily_name).read()

decompressed = gzip.decompress(daily_bytes)
inner_tar = tarfile.open(fileobj=io.BytesIO(decompressed))

target_file = "YW_2017.002_20210717_0805.asc"
inner_raw = inner_tar.extractfile(target_file).read()
print("Loaded:", target_file, "-", len(inner_raw), "bytes")

# ---- Parse header + array ----
text = inner_raw.decode("ascii")
lines = text.split("\n")
header = {}
for i in range(6):
    key, val = lines[i].split()
    header[key.lower()] = float(val)

ncols = int(header['ncols'])
nrows = int(header['nrows'])
xll = header['xllcorner']
yll = header['yllcorner']
cellsize = header['cellsize']
nodata = header['nodata_value']

data_str = " ".join(lines[6:])
values = np.array(data_str.split(), dtype=np.float32)
arr = values.reshape((nrows, ncols))
print("Parsed grid shape:", arr.shape)

# ---- Load your model grid profile ----
with rasterio.open(f"{OUT_DIR}/dtm.tif") as src:
    profile_test = src.profile.copy()
    profile_test["nodata"] = -9999.0

# ---- Diagnostic: print the small subgrid covering your domain ----
src_transform = from_origin(xll, yll + nrows * cellsize, cellsize, cellsize)

domain_west, domain_east = 414000, 422000
domain_south, domain_north = 5646000, 5654000

row_top, col_left = rowcol(src_transform, domain_west, domain_north)
row_bottom, col_right = rowcol(src_transform, domain_east, domain_south)
print(f"Source subgrid rows [{row_top}:{row_bottom}], cols [{col_left}:{col_right}]")

sub = arr[row_top:row_bottom+1, col_left:col_right+1]
print("Subgrid shape:", sub.shape)
print(sub)

Loaded: YW_2017.002_20210717_0805.asc - 4410251 bytes
Parsed grid shape: (1100, 900)
Source subgrid rows [-9313:-9305], cols [857:865]
Subgrid shape: (0, 9)
[]


In [11]:
from rasterio.warp import transform_bounds

# Convert domain bounds from EPSG:25833 (UTM 33N) into RADOLAN's native projection
radolan_west, radolan_south, radolan_east, radolan_north = transform_bounds(
    "EPSG:25833", RADOLAN_CRS,
    domain_west, domain_south, domain_east, domain_north
)
print("Domain bounds in RADOLAN's projection:")
print(f"  west={radolan_west:.1f}, east={radolan_east:.1f}")
print(f"  south={radolan_south:.1f}, north={radolan_north:.1f}")

row_top, col_left = rowcol(src_transform, radolan_west, radolan_north)
row_bottom, col_right = rowcol(src_transform, radolan_east, radolan_south)
print(f"Source subgrid rows [{row_top}:{row_bottom}], cols [{col_left}:{col_right}]")

sub = arr[row_top:row_bottom+1, col_left:col_right+1]
print("Subgrid shape:", sub.shape)
print(sub)

Domain bounds in RADOLAN's projection:
  west=276788.5, east=285828.7
  south=-4204905.6, north=-4195844.2
Source subgrid rows [537:546], cols [720:729]
Subgrid shape: (10, 10)
[[0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.1 0.  0.  0.  0.  0.  0.1 0.1 0. ]
 [0.  0.  0.  0.  0.2 0.  0.  0.1 0.1 0. ]
 [0.  0.  0.2 0.3 0.  0.  0.  0.  0.  0. ]
 [0.1 0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]]


In [13]:
import urllib.request
import gzip
import tarfile
import io
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin, rowcol
from rasterio.warp import reproject, transform_bounds, Resampling
from rasterio.crs import CRS
from datetime import datetime

CACHE_DIR = "/content/radolan_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

RADOLAN_CRS = CRS.from_proj4(
    "+proj=stere +lat_0=90 +lat_ts=60 +lon_0=10 "
    "+a=6370040 +b=6370040 +x_0=0 +y_0=0 +units=m +no_defs"
)


def download_daily_archive(date):
    year, month, day = date.year, date.month, date.day
    monthly_url = f"https://opendata.dwd.de/climate_environment/CDC/grids_germany/5_minutes/radolan/reproc/2017_002/asc/{year}/YW2017.002_{year}{month:02d}_asc.tar"
    monthly_path = f"{CACHE_DIR}/YW2017.002_{year}{month:02d}_asc.tar"

    if not os.path.exists(monthly_path):
        print(f"Downloading monthly archive for {year}-{month:02d}...")
        urllib.request.urlretrieve(monthly_url, monthly_path)

    daily_name = f"YW2017.002_{year}{month:02d}{day:02d}_asc.tar.gz"
    with tarfile.open(monthly_path) as tar:
        daily_bytes = tar.extractfile(daily_name).read()

    decompressed = gzip.decompress(daily_bytes)
    inner_tar = tarfile.open(fileobj=io.BytesIO(decompressed))
    return inner_tar


def get_event_rainfall_files(event_start, event_end):
    days_needed = sorted(set([event_start.date(), event_end.date()]))

    results = []
    for day in days_needed:
        day_dt = datetime(day.year, day.month, day.day)
        inner_tar = download_daily_archive(day_dt)
        for member in inner_tar.getnames():
            parts = member.replace(".asc", "").split("_")
            datestr, timestr = parts[-2], parts[-1]
            file_dt = datetime.strptime(datestr + timestr, "%Y%m%d%H%M")
            if event_start <= file_dt <= event_end:
                raw = inner_tar.extractfile(member).read()
                results.append((file_dt, raw))

    results.sort(key=lambda x: x[0])
    return results


def parse_radolan_asc(raw_bytes):
    text = raw_bytes.decode("ascii")
    lines = text.split("\n")
    header = {}
    for i in range(6):
        key, val = lines[i].split()
        header[key.lower()] = float(val)

    ncols = int(header['ncols']); nrows = int(header['nrows'])
    xll = header['xllcorner']; yll = header['yllcorner']
    cellsize = header['cellsize']; nodata = header['nodata_value']

    data_str = " ".join(lines[6:])
    values = np.array(data_str.split(), dtype=np.float32)
    arr = values.reshape((nrows, ncols))
    return arr, xll, yll, cellsize, nodata


def reproject_radolan_to_grid(arr, xll, yll, cellsize, nodata, dst_profile):
    nrows, ncols = arr.shape
    src_transform = from_origin(xll, yll + nrows * cellsize, cellsize, cellsize)

    dst_arr = np.full((dst_profile["height"], dst_profile["width"]),
                       dst_profile["nodata"], dtype="float32")

    reproject(
        source=arr,
        destination=dst_arr,
        src_transform=src_transform,
        src_crs=RADOLAN_CRS,
        src_nodata=nodata,
        dst_transform=dst_profile["transform"],
        dst_crs=dst_profile["crs"],
        dst_nodata=dst_profile["nodata"],
        resampling=Resampling.bilinear,
    )
    return dst_arr


def process_full_event(event_start, event_end, out_profile):
    files = get_event_rainfall_files(event_start, event_end)
    print(f"Processing {len(files)} timesteps...")

    n = len(files)
    stack = np.zeros((n, out_profile["height"], out_profile["width"]), dtype="float32")
    timestamps = []

    for i, (dt, raw) in enumerate(files):
        arr_i, xll_i, yll_i, cellsize_i, nodata_i = parse_radolan_asc(raw)
        rain_grid = reproject_radolan_to_grid(arr_i, xll_i, yll_i, cellsize_i, nodata_i, out_profile)
        rain_grid = np.where(rain_grid == out_profile["nodata"], 0.0, rain_grid)
        rain_grid = np.maximum(rain_grid, 0.0)

        stack[i] = rain_grid
        timestamps.append(dt)

        if i % 50 == 0:
            print(f"  {i}/{n}  {dt}  max={rain_grid.max():.2f}mm")

    return timestamps, stack

print("Functions loaded.")

Functions loaded.


In [14]:
with rasterio.open(f"{OUT_DIR}/dtm.tif") as src:
    profile_test = src.profile.copy()
    profile_test["nodata"] = -9999.0

event_start = datetime(2021, 7, 16, 21, 15)
event_end = datetime(2021, 7, 17, 17, 40)

timestamps, rainfall_stack = process_full_event(event_start, event_end, profile_test)
print("Final stack shape:", rainfall_stack.shape)
print("Total accumulated rain (max over domain):", rainfall_stack.sum(axis=0).max(), "mm")

Processing 246 timesteps...
  0/246  2021-07-16 21:15:00  max=3.60mm
  50/246  2021-07-17 01:25:00  max=0.00mm
  100/246  2021-07-17 05:35:00  max=0.40mm
  150/246  2021-07-17 09:45:00  max=0.00mm
  200/246  2021-07-17 13:55:00  max=1.53mm
Final stack shape: (246, 1600, 1600)
Total accumulated rain (max over domain): 37.393974 mm


In [15]:
def process_full_event_compact(event_start, event_end, out_profile, domain_bounds_utm):
    """
    Same as process_full_event, but stores only the small native-resolution
    window per timestep (compact), not the full reprojected 5m grid.
    Returns timestamps + list of (small_array, xll, yll, cellsize, nodata).
    """
    domain_west, domain_south, domain_east, domain_north = domain_bounds_utm
    radolan_west, radolan_south, radolan_east, radolan_north = transform_bounds(
        "EPSG:25833", RADOLAN_CRS, domain_west, domain_south, domain_east, domain_north
    )

    files = get_event_rainfall_files(event_start, event_end)
    print(f"Processing {len(files)} timesteps (compact storage)...")

    compact_data = []
    timestamps = []

    for i, (dt, raw) in enumerate(files):
        arr_i, xll_i, yll_i, cellsize_i, nodata_i = parse_radolan_asc(raw)
        src_transform = from_origin(xll_i, yll_i + arr_i.shape[0]*cellsize_i, cellsize_i, cellsize_i)

        row_top, col_left = rowcol(src_transform, radolan_west, radolan_north)
        row_bottom, col_right = rowcol(src_transform, radolan_east, radolan_south)
        pad = 2  # small buffer so bilinear reprojection later has neighbor context at edges
        sub = arr_i[max(0,row_top-pad):row_bottom+pad+1, max(0,col_left-pad):col_right+pad+1]

        sub_xll = xll_i + (col_left - pad) * cellsize_i
        sub_yll = yll_i + (arr_i.shape[0] - (row_bottom + pad + 1)) * cellsize_i

        compact_data.append((sub.copy(), sub_xll, sub_yll, cellsize_i, nodata_i))
        timestamps.append(dt)

        if i % 50 == 0:
            print(f"  {i}/{len(files)}  {dt}  subgrid shape={sub.shape}  max={sub.max():.2f}mm")

    return timestamps, compact_data


domain_bounds_utm = (414000, 5646000, 422000, 5654000)  # west, south, east, north

timestamps, compact_data = process_full_event_compact(
    event_start, event_end, profile_test, domain_bounds_utm
)

# Rough size check
total_floats = sum(c[0].size for c in compact_data)
print(f"\nTotal stored values: {total_floats} (~{total_floats*4/1024:.1f} KB)")
print("vs. full-grid approach would have been:", len(compact_data)*1600*1600*4/1e9, "GB")

Processing 246 timesteps (compact storage)...
  0/246  2021-07-16 21:15:00  subgrid shape=(14, 14)  max=3.60mm
  50/246  2021-07-17 01:25:00  subgrid shape=(14, 14)  max=0.00mm
  100/246  2021-07-17 05:35:00  subgrid shape=(14, 14)  max=0.40mm
  150/246  2021-07-17 09:45:00  subgrid shape=(14, 14)  max=0.00mm
  200/246  2021-07-17 13:55:00  subgrid shape=(14, 14)  max=2.20mm

Total stored values: 48216 (~188.3 KB)
vs. full-grid approach would have been: 2.51904 GB


In [16]:
import pickle

EVENTS = {
    "2017-07-09": (datetime(2017, 7, 9, 17, 55),  datetime(2017, 7, 9, 21, 15)),
    "2017-07-10": (datetime(2017, 7, 10, 15, 20), datetime(2017, 7, 10, 18, 40)),
    "2017-07-11": (datetime(2017, 7, 11, 14, 15), datetime(2017, 7, 11, 18, 5)),
    "2019-06-12": (datetime(2019, 6, 12, 16, 5),  datetime(2019, 6, 12, 22, 45)),
    "2020-08-30": (datetime(2020, 8, 30, 12, 35), datetime(2020, 8, 31, 17, 40)),
    "2021-06-29": (datetime(2021, 6, 29, 21, 5),  datetime(2021, 6, 30, 8, 10)),
    "2021-07-13": (datetime(2021, 7, 13, 18, 40), datetime(2021, 7, 14, 0, 0)),
    "2021-07-16": (datetime(2021, 7, 16, 21, 15), datetime(2021, 7, 17, 17, 40)),
}

RADOLAN_OUT_DIR = f"{OUT_DIR}/radolan_events"
os.makedirs(RADOLAN_OUT_DIR, exist_ok=True)

domain_bounds_utm = (414000, 5646000, 422000, 5654000)

for event_name, (event_start, event_end) in EVENTS.items():
    save_path = f"{RADOLAN_OUT_DIR}/{event_name}_radolan.pkl"

    if os.path.exists(save_path):
        print(f"Skipping {event_name} - already processed")
        continue

    print(f"\n=== Processing event {event_name} ===")
    timestamps, compact_data = process_full_event_compact(
        event_start, event_end, profile_test, domain_bounds_utm
    )

    with open(save_path, "wb") as f:
        pickle.dump({
            "event_name": event_name,
            "event_start": event_start,
            "event_end": event_end,
            "timestamps": timestamps,
            "compact_data": compact_data,
        }, f)

    print(f"Saved {len(timestamps)} timesteps to {save_path}")

print("\nAll events processed.")


=== Processing event 2017-07-09 ===
Processing 41 timesteps (compact storage)...
  0/41  2017-07-09 17:55:00  subgrid shape=(14, 14)  max=3.40mm
Saved 41 timesteps to /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_events/2017-07-09_radolan.pkl

=== Processing event 2017-07-10 ===
Processing 41 timesteps (compact storage)...
  0/41  2017-07-10 15:20:00  subgrid shape=(14, 14)  max=5.00mm
Saved 41 timesteps to /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_events/2017-07-10_radolan.pkl

=== Processing event 2017-07-11 ===
Processing 47 timesteps (compact storage)...
  0/47  2017-07-11 14:15:00  subgrid shape=(14, 14)  max=4.80mm
Saved 47 timesteps to /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_events/2017-07-11_radolan.pkl

=== Processing event 2019-06-12 ===
Processing 81 timesteps (compact storage)...
  0/81  2019-06-12 16:05:00  subgrid shape=(14, 14)  max=4.50mm
  50/81  2019-06-12 20:15:00  subgrid shape=(14, 14)  max=0.00mm
Saved 81 

In [17]:
import pickle

print(f"{'Event':<12} {'Steps':<7} {'Expected':<9} {'Match':<6} {'Max (mm/5min)':<14} {'Table1 Max':<10}")
print("-" * 70)

table1_max_precip = {  # from paper's Table 1, mm/5min at the gauge
    "2017-07-09": 4.2,
    "2017-07-10": 2.9,
    "2017-07-11": 7.0,
    "2019-06-12": 5.5,
    "2020-08-30": 1.8,
    "2021-06-29": 4.7,
    "2021-07-13": 8.6,
    "2021-07-16": 2.0,
}

for event_name, (event_start, event_end) in EVENTS.items():
    save_path = f"{RADOLAN_OUT_DIR}/{event_name}_radolan.pkl"
    with open(save_path, "rb") as f:
        data = pickle.load(f)

    n_steps = len(data["timestamps"])
    expected_steps = int((event_end - event_start).total_seconds() / 300) + 1
    match = "OK" if n_steps == expected_steps else "MISMATCH"

    # true max across all timesteps and all cells in the small domain window
    true_max = max(sub.max() for (sub, xll, yll, cs, nd) in data["compact_data"])

    print(f"{event_name:<12} {n_steps:<7} {expected_steps:<9} {match:<6} "
          f"{true_max:<14.2f} {table1_max_precip[event_name]:<10.1f}")

Event        Steps   Expected  Match  Max (mm/5min)  Table1 Max
----------------------------------------------------------------------
2017-07-09   41      41        OK     9.10           4.2       
2017-07-10   41      41        OK     5.00           2.9       
2017-07-11   47      47        OK     4.80           7.0       
2019-06-12   81      81        OK     8.30           5.5       
2020-08-30   350     350       OK     2.80           1.8       
2021-06-29   134     134       OK     9.00           4.7       
2021-07-13   65      65        OK     10.60          8.6       
2021-07-16   246     246       OK     4.40           2.0       
